# Fine-tune Qwen2.5-7B Instruct with QLoRA
**EduCode Rwanda — AI Coding Assistant**

Kaggle environment: 2× T4 GPU (30 hrs/week)

Stack: `transformers` + `peft` (QLoRA) + `trl` (SFTTrainer) + `bitsandbytes`

In [ ]:
%%capture
!pip install -q transformers==4.46.3 peft==0.13.2 trl==0.12.1 bitsandbytes==0.44.1 accelerate==1.1.1 datasets==3.1.0

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory // 1024**3} GB")

## 1. Load dataset
Upload `train.jsonl` and `val.jsonl` to Kaggle as a dataset and update the path below.

In [ ]:
from datasets import load_dataset

# Update this path to match your Kaggle dataset
DATA_DIR = "/kaggle/input/educode-fixjs-processed"

dataset = load_dataset(
    "json",
    data_files={"train": f"{DATA_DIR}/train.jsonl", "validation": f"{DATA_DIR}/val.jsonl"},
)
print(dataset)

## 2. Load model in 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

## 3. Apply LoRA adapters

In [ ]:
lora_config = LoraConfig(
    r=16,                        # LoRA rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Format to chat template

In [ ]:
def format_chat(example):
    """Apply Qwen chat template to each sample's messages list."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, num_proc=2)
print(dataset["train"][0]["text"][:400])

## 5. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen25-7b-educode",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,
    bf16=True,
    max_seq_length=1024,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
)

trainer.train()

## 6. Save and push to Hugging Face Hub

In [ ]:
# Save the LoRA adapter weights
SAVE_DIR = "/kaggle/working/qwen25-7b-educode-final"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

In [ ]:
# Push to Hugging Face Hub (set your HF_TOKEN in Kaggle secrets)
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
    # Replace with your HF username
    HF_REPO = "YOUR_HF_USERNAME/qwen25-7b-educode-rwanda"
    trainer.model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")
else:
    print("HF_TOKEN not set — skipping push. Download from /kaggle/working/ manually.")

## 7. Quick inference test

In [ ]:
from peft import PeftModel

# Reload for inference (merge LoRA weights)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base_model, SAVE_DIR)
merged = merged.merge_and_unload()

messages = [
    {"role": "system", "content": "You are EduCode AI, a coding assistant for Rwandan TVET students."},
    {"role": "user", "content": "Fix this JavaScript code:\n\n```javascript\nconsole.log('Hello' + x)\n```"},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(merged.device)

with torch.no_grad():
    outputs = merged.generate(**inputs, max_new_tokens=256, temperature=0.2, do_sample=True)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)